# 06·03 — LLM-as-a-Judge: Scalable, Nuanced Evaluation

> **Module:** 06 – LLM Evaluation  
> **Notebook:** 3 of 4  
> **Objective:** Build production-grade LLM evaluation systems using AI judges — understanding both their power and their well-documented failure modes.

---

## 🎯 Learning Objectives

1. Understand why LLM-as-judge emerged and what gap it fills vs BLEU/ROUGE
2. Implement the three core judge patterns: pointwise, pairwise, and reference-guided
3. Identify and quantify the known biases of LLM judges (position, verbosity, self-preference)
4. Design evaluation rubrics that produce consistent, calibrated scores
5. Build a complete evaluation pipeline with statistical analysis

---

## 1. Why LLM-as-Judge?

Traditional metrics (BLEU, ROUGE, perplexity) fail at evaluating modern LLM capabilities:

```
Task: "Explain quantum entanglement to a 10-year-old"

Output A: "Quantum entanglement is when particles become correlated so that measuring 
           one instantly tells you about the other, no matter how far apart."
           
Output B: "Imagine you have magic coins. When you and your friend flip your coins, 
           they always land opposite sides! Even if you're on different planets!"

ROUGE-1 vs some reference: A=0.42, B=0.08  (B falsely scores worse)
Human judgment:            A=3.5/5, B=4.8/5 (B is clearly better for a child!)
LLM judge:                 A=3/5,   B=5/5   (correctly evaluates audience fit)
```

LLM judges can evaluate dimensions that require **understanding**:
- Instruction following
- Factual accuracy
- Reasoning quality
- Helpfulness and tone
- Audience appropriateness
- Code correctness

---

## 2. Three Core Judge Patterns

```
┌──────────────────────────────────────────────────────────────────┐
│                    JUDGE PATTERN OVERVIEW                         │
│                                                                    │
│  POINTWISE                PAIRWISE              REFERENCE-GUIDED  │
│  ──────────               ────────              ───────────────── │
│  Score one output         Compare A vs B        Score vs gold     │
│  on absolute scale        pick better one       answer            │
│                                                                    │
│  Query → [Model] →        [Model A] → Output A  Query → [Model]  │
│  Output → [Judge]         [Model B] → Output B  Output + Ref →   │
│  → Score 1-5              → [Judge]             [Judge] → Score  │
│                           → A wins / B wins                       │
│                           / Tie                                    │
│                                                                    │
│  Pros: Simple, cheap      High sensitivity      Most accurate     │
│  Cons: Calibration hard   Order bias risk       Needs gold refs   │
└──────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install anthropic openai matplotlib seaborn pandas scipy -q

import os
import json
import re
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Literal
from scipy import stats

# Set your API key
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

print("Setup complete. Set ANTHROPIC_API_KEY to run live evaluations.")

## 3. Pattern 1: Pointwise Evaluation

Rate a single output on a defined scale.

In [ ]:
@dataclass
class EvalResult:
    score: Optional[float]
    reasoning: str
    raw_response: str
    model_used: str
    latency_ms: float
    parsed_ok: bool = True


class LLMJudge:
    """
    Production-grade LLM judge with multiple evaluation patterns.

    Design principles:
    1. Chain-of-thought reasoning BEFORE the score (more reliable)
    2. Structured output for robust parsing
    3. Explicit rubrics — reduce ambiguity in scoring
    4. Temperature=0 for reproducibility
    5. Retry logic for robustness
    """

    POINTWISE_PROMPT = """You are an expert evaluator. Your task is to assess the quality of an AI assistant's response.

## Evaluation Criteria
{criteria}

## Scoring Rubric
1 - Very Poor: Fails to address the question, contains major errors, or is completely unhelpful
2 - Poor: Partially addresses the question but has significant gaps or errors
3 - Acceptable: Addresses the question adequately with minor issues
4 - Good: Clear, accurate, and helpful response with only trivial issues
5 - Excellent: Comprehensive, accurate, perfectly tailored response

## Question
{question}

## Response to Evaluate
{response}

## Instructions
First, analyze the response step by step against each criterion.
Then provide your final assessment.

Respond in this exact JSON format:
{{
  "analysis": "<your step-by-step analysis>",
  "strengths": ["<strength 1>", "<strength 2>"],
  "weaknesses": ["<weakness 1>"],
  "score": <integer 1-5>,
  "score_rationale": "<one sentence justifying the score>"
}}"""

    PAIRWISE_PROMPT = """You are an expert evaluator comparing two AI assistant responses.

## Evaluation Criteria
{criteria}

## Question
{question}

## Response A
{response_a}

## Response B
{response_b}

## Instructions
Compare both responses carefully against each criterion.
Be objective — ignore length and style differences that don't affect quality.

Respond in this exact JSON format:
{{
  "analysis": "<compare A and B on each criterion>",
  "a_strengths": ["<strength>"],
  "b_strengths": ["<strength>"],
  "winner": "A" | "B" | "tie",
  "confidence": "high" | "medium" | "low",
  "rationale": "<one sentence explaining the decision>"
}}"""

    REFERENCE_GUIDED_PROMPT = """You are an expert evaluator. Compare a model response to the ideal reference answer.

## Evaluation Criteria
{criteria}

## Question
{question}

## Reference Answer (Gold Standard)
{reference}

## Model Response
{response}

## Instructions
Compare the model response to the reference. The model doesn't need to match word-for-word —
award full credit for semantically equivalent correct answers.
Penalise factual errors, missing key points, or incorrect reasoning.

Respond in this exact JSON format:
{{
  "analysis": "<compare response to reference point by point>",
  "key_points_covered": ["<point>"],
  "key_points_missing": ["<point>"],
  "factual_errors": ["<error or 'none'>"],
  "score": <integer 1-5>,
  "score_rationale": "<why this score>"
}}"""

    def __init__(self, model: str = "claude-3-haiku-20240307", api_key: Optional[str] = None):
        self.model = model
        self.api_key = api_key or os.environ.get("ANTHROPIC_API_KEY")
        self._client = None

    @property
    def client(self):
        if self._client is None:
            import anthropic
            self._client = anthropic.Anthropic(api_key=self.api_key)
        return self._client

    def _call(self, prompt: str, max_retries: int = 3) -> Tuple[str, float]:
        """Call the judge model with retry logic."""
        for attempt in range(max_retries):
            try:
                t0 = time.perf_counter()
                response = self.client.messages.create(
                    model=self.model,
                    max_tokens=1024,
                    temperature=0,  # Deterministic for reproducibility
                    messages=[{"role": "user", "content": prompt}]
                )
                latency = (time.perf_counter() - t0) * 1000
                return response.content[0].text, latency
            except Exception as e:
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)

    def _parse_json(self, text: str) -> Tuple[Dict, bool]:
        """Robustly extract JSON from model response."""
        # Try direct parse
        try:
            return json.loads(text.strip()), True
        except:
            pass
        # Extract from markdown code block
        match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
        if match:
            try:
                return json.loads(match.group(1).strip()), True
            except:
                pass
        # Find any JSON object in the text
        match = re.search(r'\{[\s\S]*\}', text)
        if match:
            try:
                return json.loads(match.group(0)), True
            except:
                pass
        return {}, False

    def pointwise(
        self,
        question: str,
        response: str,
        criteria: str = "Accuracy, Helpfulness, Clarity, Completeness",
    ) -> EvalResult:
        prompt = self.POINTWISE_PROMPT.format(
            criteria=criteria, question=question, response=response
        )
        raw, latency = self._call(prompt)
        parsed, ok = self._parse_json(raw)
        return EvalResult(
            score=parsed.get("score"),
            reasoning=parsed.get("analysis", "") + " | " + parsed.get("score_rationale", ""),
            raw_response=raw,
            model_used=self.model,
            latency_ms=latency,
            parsed_ok=ok,
        )

    def pairwise(
        self,
        question: str,
        response_a: str,
        response_b: str,
        criteria: str = "Accuracy, Helpfulness, Clarity, Completeness",
        swap_and_average: bool = True,
    ) -> Dict:
        """
        Compare two responses.

        swap_and_average=True runs the comparison twice (A vs B, then B vs A)
        to cancel out position bias. This is the recommended approach.
        """
        def _run(a, b, label):
            prompt = self.PAIRWISE_PROMPT.format(
                criteria=criteria, question=question, response_a=a, response_b=b
            )
            raw, latency = self._call(prompt)
            parsed, ok = self._parse_json(raw)
            return parsed, raw, latency

        result_ab, raw_ab, lat_ab = _run(response_a, response_b, "A vs B")
        winner_ab = result_ab.get("winner", "tie")

        if not swap_and_average:
            return {"winner": winner_ab, "confidence": result_ab.get("confidence"),
                    "analysis": result_ab.get("analysis"), "latency_ms": lat_ab}

        result_ba, raw_ba, lat_ba = _run(response_b, response_a, "B vs A")
        winner_ba_raw = result_ba.get("winner", "tie")  # in BA order
        # Flip: if judge says B wins when B is presented as A, that means original A wins
        winner_ba = {"A": "B", "B": "A", "tie": "tie"}.get(winner_ba_raw, "tie")

        # Aggregate
        if winner_ab == winner_ba:
            final_winner = winner_ab
            consistency = "consistent"
        else:
            final_winner = "tie"  # inconsistent → call it a tie
            consistency = "inconsistent"

        return {
            "winner": final_winner,
            "winner_ab": winner_ab,
            "winner_ba": winner_ba,
            "consistency": consistency,
            "analysis_ab": result_ab.get("analysis"),
            "total_latency_ms": lat_ab + lat_ba,
        }

    def reference_guided(
        self,
        question: str,
        response: str,
        reference: str,
        criteria: str = "Accuracy, Completeness, Factual Correctness",
    ) -> EvalResult:
        prompt = self.REFERENCE_GUIDED_PROMPT.format(
            criteria=criteria, question=question, reference=reference, response=response
        )
        raw, latency = self._call(prompt)
        parsed, ok = self._parse_json(raw)
        return EvalResult(
            score=parsed.get("score"),
            reasoning=parsed.get("analysis", ""),
            raw_response=raw,
            model_used=self.model,
            latency_ms=latency,
            parsed_ok=ok,
        )


print("LLMJudge class ready. Initialize with your API key to run evaluations.")
print("\nExample usage:")
print("  judge = LLMJudge('claude-3-haiku-20240307')")
print("  result = judge.pointwise(question, response)")

In [ ]:
# ── Mock judge for offline development / testing ──────────────────────────
# Use this when you don't have an API key to explore the rest of the notebook

class MockJudge:
    """Mock judge that simulates realistic LLM judge behavior for testing."""

    def __init__(self, noise_std: float = 0.3):
        self.noise_std = noise_std

    def pointwise(self, question: str, response: str, criteria: str = "") -> EvalResult:
        # Simulate score based on response length and content keywords
        quality_signals = [
            len(response) > 100,
            any(kw in response.lower() for kw in ['because', 'however', 'therefore', 'specifically']),
            response.count('.') >= 2,
            not any(bad in response.lower() for bad in ['idk', 'dunno', 'whatever']),
        ]
        base_score = 2 + sum(quality_signals)
        noisy_score = np.clip(base_score + np.random.normal(0, self.noise_std), 1, 5)
        return EvalResult(
            score=round(noisy_score, 1),
            reasoning=f"[Mock] Response quality assessed. Base signal: {sum(quality_signals)}/4",
            raw_response=json.dumps({"score": round(noisy_score), "analysis": "mock"}),
            model_used="mock",
            latency_ms=np.random.exponential(200),
        )

    def pairwise(self, question, response_a, response_b, **kwargs):
        score_a = self.pointwise(question, response_a).score
        score_b = self.pointwise(question, response_b).score
        if abs(score_a - score_b) < 0.3:
            winner = "tie"
        else:
            winner = "A" if score_a > score_b else "B"
        return {"winner": winner, "score_a": score_a, "score_b": score_b,
                "consistency": "consistent", "total_latency_ms": 400}


# Switch between real and mock judge
USE_REAL_API = bool(os.environ.get("ANTHROPIC_API_KEY"))
judge = LLMJudge("claude-3-haiku-20240307") if USE_REAL_API else MockJudge()
print(f"Using: {'Real Claude judge' if USE_REAL_API else 'Mock judge (set ANTHROPIC_API_KEY for real evaluation)'}")

In [ ]:
# ── Test pointwise evaluation ──────────────────────────────────────────────
eval_cases = [
    {
        "question": "Explain the difference between supervised and unsupervised learning.",
        "responses": [
            ("Excellent detailed",
             """Supervised learning uses labeled data — each training example has both an input 
             and a correct output (label). The model learns to predict labels for new inputs. 
             Examples: spam detection, image classification.
             
             Unsupervised learning finds patterns in unlabeled data without explicit guidance. 
             The model discovers structure on its own. Examples: clustering customers by behavior, 
             topic modeling in documents, anomaly detection.
             
             Key difference: supervised has 'right answers' to learn from; unsupervised must 
             discover structure independently."""),

            ("Vague and short",
             "Supervised uses labels, unsupervised doesn't. Both are types of machine learning."),

            ("Factually wrong",
             """Supervised learning is faster because it uses GPUs. Unsupervised learning is when 
             humans supervise the model manually. Both require at least 1 million training examples 
             to work properly."""),

            ("Good but off-topic",
             """Machine learning has many applications in healthcare, finance, and technology. 
             Companies like Google and Amazon use it extensively. Neural networks are a popular 
             approach. You should definitely learn Python to work in this field."""),
        ]
    }
]

criteria = """1. Accuracy: Is the explanation factually correct?
2. Completeness: Are both concepts adequately explained?
3. Clarity: Is it easy to understand?
4. Relevance: Does it directly answer the question?"""

print("Pointwise Evaluation Results:")
print(f"Question: {eval_cases[0]['question']}")
print()

results = []
for response_label, response_text in eval_cases[0]['responses']:
    result = judge.pointwise(eval_cases[0]['question'], response_text, criteria)
    results.append((response_label, result))
    print(f"[{response_label}]")
    print(f"  Score: {result.score}/5")
    print(f"  Reasoning: {result.reasoning[:120]}..." if len(result.reasoning) > 120 else f"  Reasoning: {result.reasoning}")
    print(f"  Latency: {result.latency_ms:.0f}ms")
    print()

## 4. Known Biases in LLM Judges

In [ ]:
# ── Bias 1: Verbosity Bias ─────────────────────────────────────────────────
# LLM judges often prefer longer responses, even when shorter is better.

verbosity_test = [
    ("Concise correct",
     "The capital of France is Paris."),

    ("Verbose correct (same info)",
     """That's a great question! I'd be happy to help with that. 
     The capital city of the beautiful country of France, which is located in Western Europe 
     and is known for the Eiffel Tower, excellent cuisine, and rich cultural history, 
     is indeed Paris. Paris is also known as the City of Light and has been the capital 
     of France for many centuries. I hope that answers your question!"""),

    ("Medium length correct",
     "The capital of France is Paris. It's been the country's capital since the 10th century "
     "and is home to the Eiffel Tower and Louvre museum."),
]

question = "What is the capital of France?"
print("Verbosity Bias Test:")
print(f"Question: {question}")
print()
print(f"{'Response Type':<30} {'Words':>6} {'Score':>6}")
print("-" * 45)
for label, resp in verbosity_test:
    r = judge.pointwise(question, resp, "Accuracy and helpfulness")
    word_count = len(resp.split())
    print(f"{label:<30} {word_count:>6} {str(r.score):>6}")

print()
print("Insight: All three responses are equally correct, but verbose responses")
print("often receive higher scores from LLM judges. This is documented in")
print("Zheng et al. (2023) - MT-Bench paper.")

In [ ]:
# ── Bias 2: Position Bias in Pairwise Evaluation ───────────────────────────
# LLM judges often prefer whichever response comes first (or second).
# Mitigation: Run A-B and B-A, check consistency.

position_bias_test = [
    {
        "question": "What causes inflation?",
        "response_a": "Inflation is caused by too much money chasing too few goods. "
                      "When the money supply grows faster than economic output, prices rise. "
                      "Common causes include central bank policy, supply chain disruptions, and strong demand.",
        "response_b": "Inflation occurs when the general price level in an economy rises over time. "
                      "Key drivers include demand-pull factors (high consumer spending), "
                      "cost-push factors (rising production costs), and monetary expansion.",
        "quality": "roughly_equal",
    },
]

print("Position Bias Experiment:")
print("(Running each comparison twice with swapped order)")
print()

N_TRIALS = 5  # Run multiple times to measure consistency
wins = {"A": 0, "B": 0, "tie": 0}
consistency_count = 0

ex = position_bias_test[0]
for i in range(N_TRIALS):
    result = judge.pairwise(
        ex['question'], ex['response_a'], ex['response_b'],
        swap_and_average=True
    )
    wins[result['winner']] += 1
    if result.get('consistency') == 'consistent':
        consistency_count += 1
    print(f"Trial {i+1}: winner={result['winner']}, consistency={result.get('consistency', 'N/A')}")

print(f"\nResults over {N_TRIALS} trials:")
print(f"  A wins: {wins['A']}/{N_TRIALS}")
print(f"  B wins: {wins['B']}/{N_TRIALS}")
print(f"  Ties:   {wins['tie']}/{N_TRIALS}")
print(f"  Consistent (A-B agrees with B-A): {consistency_count}/{N_TRIALS}")
print()
print("High inconsistency = evidence of position bias or random variation")
print("Low inconsistency = judge has stable preferences")

In [ ]:
# ── Bias 3: Self-Preference / Model Affiliation Bias ──────────────────────
# LLM judges have been shown to prefer outputs from models in the same family.
# Mitigation: Use a judge from a DIFFERENT model family than what you're evaluating.

# Visualize the bias taxonomy
biases = {
    "Verbosity bias": {
        "description": "Prefers longer responses even if quality is equal",
        "severity": 4,
        "mitigation": "Instruct judge to evaluate quality, not length",
    },
    "Position bias": {
        "description": "Prefers response presented first or second",
        "severity": 4,
        "mitigation": "Run pairwise both ways and check consistency",
    },
    "Self-preference": {
        "description": "Prefers outputs from its own model family",
        "severity": 3,
        "mitigation": "Use judge from different model family",
    },
    "Formatting bias": {
        "description": "Prefers responses with markdown, headers, bullets",
        "severity": 3,
        "mitigation": "Normalize formatting before evaluation",
    },
    "Sycophancy": {
        "description": "Aligns score with user's expected/desired answer",
        "severity": 2,
        "mitigation": "Blind evaluation (hide which model produced each response)",
    },
    "Anchoring": {
        "description": "First score in a batch anchors subsequent scores",
        "severity": 2,
        "mitigation": "Randomize evaluation order",
    },
    "Domain unfamiliarity": {
        "description": "Cannot assess correctness in highly specialized domains",
        "severity": 5,
        "mitigation": "Use domain-specific judge or human expert evaluation",
    },
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Severity chart
ax = axes[0]
names = list(biases.keys())
severities = [b['severity'] for b in biases.values()]
colors = ['#d62728' if s >= 4 else '#ff7f0e' if s == 3 else '#2ca02c' for s in severities]
bars = ax.barh(names, severities, color=colors, alpha=0.85)
ax.bar_label(bars, labels=[f'{s}/5' for s in severities], padding=3, fontsize=10)
ax.set_xlim(0, 6.5)
ax.set_xlabel('Severity (1=minor, 5=critical)', fontsize=11)
ax.set_title('Known LLM Judge Biases by Severity', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='x')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#d62728', label='High (4-5)'),
    Patch(facecolor='#ff7f0e', label='Medium (3)'),
    Patch(facecolor='#2ca02c', label='Low (1-2)'),
]
ax.legend(handles=legend_elements, loc='lower right')

# Mitigation strategies
ax = axes[1]
ax.axis('off')
table_data = [[name[:20], b['mitigation'][:40]] for name, b in biases.items()]
table = ax.table(
    cellText=table_data,
    colLabels=['Bias', 'Mitigation'],
    cellLoc='left',
    loc='center',
    bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(9)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2196F3')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#f5f5f5')
ax.set_title('Mitigation Strategies', fontsize=12, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('../figures/06_judge_biases.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Building a Production Evaluation Pipeline

In [ ]:
class EvaluationPipeline:
    """
    Full evaluation pipeline for comparing multiple models on a test set.
    
    Features:
    - Multi-dimensional scoring (accuracy, helpfulness, safety, etc.)
    - Statistical significance testing
    - Automated report generation
    - Cost estimation
    """

    def __init__(self, judge: LLMJudge, dimensions: Optional[List[str]] = None):
        self.judge = judge
        self.dimensions = dimensions or [
            "Accuracy",
            "Helpfulness",
            "Clarity",
            "Completeness",
        ]

    def evaluate_dataset(
        self,
        test_cases: List[Dict],
        model_responses: Dict[str, List[str]],
    ) -> pd.DataFrame:
        """
        Evaluate multiple models on all test cases.

        Args:
            test_cases: List of {"question": str, "reference": str (optional)}
            model_responses: {model_name: [response_1, response_2, ...]}

        Returns:
            DataFrame with columns: model, question_id, score, latency_ms, dimension
        """
        records = []
        criteria = "\n".join(f"- {d}" for d in self.dimensions)

        for q_idx, tc in enumerate(test_cases):
            for model_name, responses in model_responses.items():
                if q_idx >= len(responses):
                    continue
                response = responses[q_idx]
                result = self.judge.pointwise(tc['question'], response, criteria)
                records.append({
                    'model': model_name,
                    'question_id': q_idx,
                    'question': tc['question'][:60],
                    'score': result.score,
                    'latency_ms': result.latency_ms,
                    'parsed_ok': result.parsed_ok,
                })

        return pd.DataFrame(records)

    def statistical_analysis(self, df: pd.DataFrame) -> Dict:
        """Compute statistical significance of score differences between models."""
        models = df['model'].unique()
        results = {}

        for m in models:
            scores = df[df['model'] == m]['score'].dropna()
            results[m] = {
                'mean': round(scores.mean(), 3),
                'std': round(scores.std(), 3),
                'median': round(scores.median(), 3),
                'n': len(scores),
                'ci_95': tuple(round(x, 3) for x in stats.t.interval(
                    0.95, df=len(scores)-1,
                    loc=scores.mean(),
                    scale=stats.sem(scores)
                )) if len(scores) > 1 else (None, None),
            }

        # Pairwise t-tests
        sig_tests = {}
        model_list = list(models)
        for i in range(len(model_list)):
            for j in range(i+1, len(model_list)):
                m1, m2 = model_list[i], model_list[j]
                s1 = df[df['model'] == m1]['score'].dropna()
                s2 = df[df['model'] == m2]['score'].dropna()
                if len(s1) > 1 and len(s2) > 1:
                    _, p_val = stats.ttest_ind(s1, s2)
                    sig_tests[f"{m1} vs {m2}"] = {
                        'p_value': round(p_val, 4),
                        'significant': p_val < 0.05,
                        'effect_size_d': round((s1.mean() - s2.mean()) / (
                            np.sqrt((s1.std()**2 + s2.std()**2) / 2) + 1e-9
                        ), 3),
                    }

        return {'model_stats': results, 'significance_tests': sig_tests}

    def plot_results(self, df: pd.DataFrame, title: str = "Model Evaluation Results"):
        fig, axes = plt.subplots(1, 3, figsize=(16, 6))

        # Score distributions
        ax = axes[0]
        for model in df['model'].unique():
            scores = df[df['model'] == model]['score'].dropna()
            ax.hist(scores, alpha=0.6, bins=np.arange(0.5, 6, 1), label=model, density=True)
        ax.set_xlabel('Score')
        ax.set_ylabel('Density')
        ax.set_title('Score Distributions')
        ax.set_xticks([1, 2, 3, 4, 5])
        ax.legend()
        ax.grid(alpha=0.3)

        # Box plots
        ax = axes[1]
        models = df['model'].unique()
        data = [df[df['model'] == m]['score'].dropna().tolist() for m in models]
        bp = ax.boxplot(data, labels=models, patch_artist=True)
        colors = plt.cm.Set2(np.linspace(0, 1, len(models)))
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        ax.set_ylabel('Score (1-5)')
        ax.set_title('Score Distribution by Model')
        ax.grid(alpha=0.3, axis='y')

        # Mean scores with confidence intervals
        ax = axes[2]
        analysis = self.statistical_analysis(df)
        model_names = list(analysis['model_stats'].keys())
        means = [analysis['model_stats'][m]['mean'] for m in model_names]
        cis = [analysis['model_stats'][m]['ci_95'] for m in model_names]
        yerrs = [(m - ci[0], ci[1] - m) if ci[0] else (0, 0)
                 for m, ci in zip(means, cis)]
        lower = [ye[0] for ye in yerrs]
        upper = [ye[1] for ye in yerrs]

        ax.bar(model_names, means,
               yerr=[lower, upper],
               capsize=5, color=colors[:len(model_names)], alpha=0.85)
        ax.set_ylabel('Mean Score (with 95% CI)')
        ax.set_title('Model Comparison (Mean ± 95% CI)')
        ax.set_ylim(0, 5.5)
        ax.grid(alpha=0.3, axis='y')

        plt.suptitle(title, fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('../figures/06_eval_results.png', dpi=150, bbox_inches='tight')
        plt.show()


# ── Run a synthetic evaluation ─────────────────────────────────────────────
test_set = [
    {"question": "What is the difference between RAM and ROM?"},
    {"question": "Explain gradient descent in simple terms."},
    {"question": "What causes a stack overflow error?"},
    {"question": "What is the CAP theorem?"},
    {"question": "Explain how HTTPS encryption works."},
]

# Simulated model responses (in practice, these come from your LLM APIs)
model_responses = {
    "ModelA_7B": [
        "RAM is temporary memory used when the computer is running. ROM is permanent memory that stores firmware.",
        "Gradient descent is an optimization algorithm. It finds the minimum of a function by iteratively moving in the direction of steepest descent.",
        "A stack overflow occurs when a program calls too many functions recursively and runs out of stack space.",
        "CAP theorem states distributed systems can only guarantee two of: Consistency, Availability, Partition tolerance.",
        "HTTPS uses SSL/TLS to encrypt data between browser and server.",
    ],
    "ModelB_13B": [
        "RAM (Random Access Memory) is volatile memory that stores data your computer is actively using — it's fast but lost when power is off. ROM (Read-Only Memory) is non-volatile memory containing permanent data like firmware, which persists without power.",
        "Gradient descent is like rolling a ball down a hill to find the lowest point. The algorithm computes the gradient (slope) of the loss function and takes small steps in the opposite direction. The learning rate controls step size. It iterates until convergence.",
        "A stack overflow happens when the call stack exceeds its memory limit — most commonly from infinite recursion (a function calling itself endlessly). Each function call adds a frame to the stack; too many calls exhausts available stack memory.",
        "The CAP theorem proves distributed systems can guarantee only two of three properties simultaneously: Consistency (every read gets the latest write), Availability (every request gets a response), and Partition tolerance (system works despite network failures). Since partitions happen in real systems, you typically choose CP or AP.",
        "HTTPS works by establishing a TLS session: the server sends its certificate, the client verifies it, they perform a key exchange to establish a shared secret, then all data is symmetrically encrypted. The certificate proves server identity; encryption prevents eavesdropping.",
    ],
}

pipeline = EvaluationPipeline(MockJudge())
df = pipeline.evaluate_dataset(test_set, model_responses)

print("Evaluation complete. Results:")
print(df.groupby('model')['score'].agg(['mean', 'std', 'count']).round(3))

pipeline.plot_results(df, "ModelA_7B vs ModelB_13B Evaluation")

stats_result = pipeline.statistical_analysis(df)
print("\nStatistical Analysis:")
for model, s in stats_result['model_stats'].items():
    print(f"  {model}: mean={s['mean']}, 95%CI={s['ci_95']}")
for comparison, test in stats_result['significance_tests'].items():
    print(f"  {comparison}: p={test['p_value']}, significant={test['significant']}, d={test['effect_size_d']}")

## 6. Designing Rubrics: The Most Important Skill

The quality of your evaluation is only as good as your rubric. Vague criteria produce noisy, inconsistent scores.

In [ ]:
# Compare vague vs precise rubrics
vague_rubric = "Rate the quality of this response from 1 to 5."

precise_rubric = """
Rate this response from 1 to 5 using these EXACT criteria:

Score 5 (Excellent):
  - Directly and completely answers the specific question asked
  - All factual claims are accurate and verifiable
  - Appropriate detail level for the question complexity
  - No unnecessary filler, hedging, or off-topic content

Score 4 (Good):
  - Answers the question with only trivial omissions
  - Factually accurate with at most one minor imprecision
  - Slightly over- or under-detailed

Score 3 (Acceptable):
  - Partially answers the question with notable gaps
  - May contain minor factual inaccuracies
  - Or significantly over/under-detailed

Score 2 (Poor):
  - Minimally addresses the question
  - Contains significant factual errors
  - Or is so vague as to be unhelpful

Score 1 (Very Poor):
  - Does not address the question at all
  - Contains completely wrong information
  - Or is harmful/inappropriate

IMPORTANT: Do not award higher scores for longer responses.
           A concise accurate answer beats a verbose inaccurate one.
"""

test_responses = [
    "Photosynthesis converts sunlight, CO₂, and water into glucose and oxygen in plant cells.",
    "Photosynthesis is very important for plants and helps them grow. It happens in the leaves.",
    "I think photosynthesis might involve some kind of chemical process but I'm not entirely sure about the details.",
]

question = "What is photosynthesis?"

print("Rubric Quality Comparison:")
print(f"{'Response':<55} {'Vague':>7} {'Precise':>8}")
print("-" * 72)
for resp in test_responses:
    score_vague   = judge.pointwise(question, resp, vague_rubric).score
    score_precise = judge.pointwise(question, resp, precise_rubric).score
    print(f"{resp[:55]:<55} {str(score_vague):>7} {str(score_precise):>8}")

print()
print("Precise rubrics produce more consistent and reliable scores.")
print("The key is defining exactly what each score level means.")

## 7. Engineering Notes

### Production Checklist

```python
# ✅ Always do:
# 1. Use temperature=0 for reproducibility
# 2. Request chain-of-thought BEFORE the score (not after)
# 3. Explicitly define each score level in the rubric
# 4. Run pairwise comparisons in both orders (A-B and B-A)
# 5. Use a stronger model as judge (e.g., Claude 3.5 Sonnet judges Claude Haiku)
# 6. Sample 10-20% of results for human spot-checking
# 7. Report confidence intervals, not just means
# 8. Version-control your prompts and rubrics

# ❌ Avoid:
# 1. Using the same model to generate AND judge its own outputs
# 2. Rubrics with only numeric scales and no descriptions
# 3. Evaluating highly specialized domains without human expert calibration
# 4. Treating LLM judge scores as ground truth
# 5. Aggregating scores without statistical significance testing
```

### Cost Estimation

```
LLM evaluation cost for 1000 test cases:
  Prompt tokens:  ~500 tokens/case × 1000 = 500K tokens
  Output tokens:  ~200 tokens/case × 1000 = 200K tokens

  Claude 3 Haiku: $0.25 input + $1.25 output per 1M tokens
  Cost:           500K × $0.25/1M + 200K × $1.25/1M = ~$0.37 total

  Claude 3.5 Sonnet: $3 input + $15 output per 1M tokens
  Cost:           500K × $3/1M + 200K × $15/1M = ~$4.50 total
```

## 8. Exercises

1. **Verbosity Audit**: Take any 20 model responses, measure word count and judge score. Compute the Pearson correlation. How strong is verbosity bias in your judge?

2. **Rubric Ablation**: Evaluate 10 responses using: (a) no rubric, (b) 3-level rubric, (c) 5-level rubric with descriptions. Compute score variance. Which rubric produces most consistent scores?

3. **Inter-Rater Reliability**: Have Claude and GPT-4 both score the same 20 responses. Compute Cohen's Kappa. At what threshold do they agree?

4. **Human Calibration**: Take 20 LLM judge scores and get human judgments for the same items. Compute Spearman rank correlation. Is your judge calibrated to humans?

5. **Full MT-Bench**: Implement a subset of the MT-Bench evaluation suite (multi-turn conversations). Compare 2 models using pairwise comparison with bias mitigation.

---

## 9. References

- [Zheng et al. (2023) — Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685)
- [Wang et al. (2023) — Large Language Models are not Fair Evaluators](https://arxiv.org/abs/2305.17926)
- [Dubois et al. (2024) — AlpacaEval: An Automatic Evaluator of Instruction-Following Models](https://github.com/tatsu-lab/alpaca_eval)
- [Chiang et al. (2024) — Chatbot Arena: An Open Platform for Evaluating LLMs by Human Preference](https://arxiv.org/abs/2403.04132)